First we load the datasets and visualize what they look like to think how to use them. 

In [1]:
#imports
import pandas as pd

In [2]:
#load the cleaned datasets
booking_requests = pd.read_csv('cleaned_data/booking_requests.csv')
cleaners = pd.read_csv('cleaned_data/cleaners.csv')
customers = pd.read_csv('cleaned_data/customers.csv')
marketing_spend = pd.read_csv('cleaned_data/marketing_spend.csv')

In [3]:
#visualize the datasets

booking_requests


,request_id,customer_id,city,request_date,hours,price_eur,status,cleaner_id,rating,fulfilled,canc,month
0,R0000609,C000609,Lyon,2025-01-14,4,108.0,fulfilled,P00037,5.0,True,False,2025-01-01
1,R0010556,C002995,Bordeaux,2025-05-07,3,81.0,fulfilled,P00186,4.0,True,False,2025-05-01
2,R0001916,C001191,Lyon,2025-02-20,3,81.0,unfulfilled,NaN,NaN,False,False,2025-02-01
3,R0061885,C009288,Paris,2026-01-14,3,81.0,fulfilled,P00212,3.0,True,False,2026-01-01
4,R0015685,C000933,Paris,2025-07-12,3,81.0,fulfilled,P00136,4.0,True,False,2025-07-01
...,...,...,...,...,...,...,...,...,...,...,...,...
132159,R0038441,C004736,Lyon,2025-10-24,4,108.0,fulfilled,P00234,5.0,True,False,2025-10-01
132160,R0040942,C009999,Bordeaux,2025-10-25,3,81.0,fulfilled,P00170,5.0,True,False,2025-10-01
132161,R0027337,C002467,Paris,2025-09-21,2,54.0,fulfilled,P00501,5.0,True,False,2025-09-01
132162,R0106418,C010209,Paris,2026-05-20,3,81.0,fulfilled,P01096,5.0,True,False,2026-05-01


In [4]:
#visualize the datasets
cleaners

,cleaner_id,application_date,city,acquisition_channel,onboarded_date,deactivated_date
0,P00027,2025-01-01,Paris,Meta Ads,2025-01-19,2025-01-30
1,P00002,2025-01-01,Paris,Meta Ads,2025-01-10,2025-12-31
2,P00028,2025-01-01,Paris,Job boards,2025-01-10,2025-08-24
3,P00014,2025-01-02,Paris,Meta Ads,2025-01-19,NaN
4,P00025,2025-01-04,Paris,Job boards,2025-01-23,2025-10-23
...,...,...,...,...,...,...
1506,P01504,2026-06-29,Lille,Meta Ads,2026-07-12,NaN
1507,P01461,2026-06-29,Bordeaux,Job boards,2026-07-13,NaN
1508,P01506,2026-06-30,Lille,Referral,NaN,NaN
1509,P01453,2026-06-30,Lyon,Job boards,2026-07-20,NaN


In [5]:
customers


,customer_id,signup_date,city,acquisition_channel,year,month,day
0,C000192,2025-01-01,Paris,Google Ads,2025,1,1
1,C000039,2025-01-01,Paris,Google Ads,2025,1,1
2,C000572,2025-01-01,Lyon,Organic,2025,1,1
3,C000504,2025-01-01,Lyon,Meta Ads,2025,1,1
4,C000307,2025-01-01,Paris,Google Ads,2025,1,1
...,...,...,...,...,...,...,...
24627,C024347,2026-06-30,Bordeaux,Meta Ads,2026,6,30
24628,C022608,2026-06-30,Paris,Organic,2026,6,30
24629,C024561,2026-06-30,Lille,Google Ads,2026,6,30
24630,C024436,2026-06-30,Lille,Google Ads,2026,6,30


In [6]:
marketing_spend

,month,city,side,channel,spend_eur
0,2025-01-01,Paris,demand,Google Ads,3920.0
1,2025-01-01,Paris,demand,Meta Ads,2880.0
2,2025-01-01,Paris,supply,Job boards,1620.0
3,2025-01-01,Paris,supply,Meta Ads,480.0
4,2025-01-01,Lyon,demand,Google Ads,2290.0
...,...,...,...,...,...
238,2026-06-01,Bordeaux,supply,Meta Ads,1750.0
239,2026-06-01,Lille,demand,Google Ads,2450.0
240,2026-06-01,Lille,demand,Meta Ads,1440.0
241,2026-06-01,Lille,supply,Job boards,770.0


The first thing we remark is that there are two funnels, one for the client, one for the cleaners.
Let's first work on the acquisition of both.

I- Acquisition 

A- Customers


In [7]:
#calculate the number of customers by acquisition channel
customers.groupby('acquisition_channel').size().reset_index(name='count').sort_values(by='count', ascending=False)

,acquisition_channel,count
0,Google Ads,8589
1,Meta Ads,7362
2,Organic,6177
3,Referral,2504


B- Cleaners

In [8]:
#calculate the number of cleaners by acquisition channel
cleaners.groupby('acquisition_channel').size().reset_index(name='count').sort_values(by='count', ascending=False)

,acquisition_channel,count
0,Job boards,818
1,Meta Ads,488
2,Referral,205


II- Engaging 

A- Clients

In [9]:
#number of bookings by customer
booking_requests.groupby('customer_id').size().reset_index(name='count').sort_values(by='count', ascending=False)

,customer_id,count
817,C000818,44
135,C000136,44
2869,C002870,37
859,C000860,37
1316,C001317,36
...,...,...
24403,C024536,1
24404,C024537,1
24405,C024538,1
24406,C024539,1


In [10]:
#number of customers with no bookings
noBooking = customers[~customers['customer_id'].isin(booking_requests['customer_id'])].shape[0]
print(noBooking)


141


In [11]:
#customers with only uncompleted bookings
customersWithUncompletedBookings = customers[customers['customer_id'].isin(booking_requests[booking_requests['status'] == 'unfulfilled']['customer_id'])]
customersWithUncompletedBookings.shape[0]


9196

In [12]:
#Do those customers have any completed bookings?
customersWithUncompletedBookings[customersWithUncompletedBookings['customer_id'].isin(booking_requests[booking_requests['status'] == 'completed']['customer_id'])].shape[0]

0

We remark that there is a small number of customers which do not do any booking and a huge number of them who try to do a booking but it is not fullfilled, they are not activated...

In [13]:
#calculate the pourcentage of activation
pourcentage_activation = (customers.shape[0]- noBooking - customersWithUncompletedBookings.shape[0]) / customers.shape[0] * 100
print(f"Pourcentage of activation: {pourcentage_activation:.2f}%")

Pourcentage of activation: 62.09%


Around 62% of the clients activate the service.

We would now like to know which part of them is kept (retention).

The first step is to keep only the transaction which have succeeded. 

In [14]:
fullfiled_requests = booking_requests[booking_requests['status'] == 'fulfilled']
fullfiled_requests.groupby('customer_id').size().reset_index(name='count').sort_values(by='count', ascending=False)

,customer_id,count
649,C000818,42
127,C000136,39
1363,C001632,34
689,C000860,34
2103,C002573,33
...,...,...
20569,C024587,1
20570,C024588,1
20571,C024589,1
20572,C024591,1


In [15]:
#get the number of clients for each number of fulfilled bookings
fulfilled_bookings_count = fullfiled_requests.groupby('customer_id').size().reset_index(name='count')
fulfilled_bookings_distribution = fulfilled_bookings_count.groupby('count').size().reset_index(name='number_of_customers').sort_values(by='count', ascending=False)
fulfilled_bookings_distribution

,count,number_of_customers
35,42,1
34,39,1
33,34,2
32,33,5
31,32,7
30,31,6
29,30,12
28,29,11
27,28,18
26,27,23


For the customers with only 1 command, what is the reason? Is the last command recent?

In [16]:
#check the date of the last booking for clients with only one fulfilled booking
last_booking_date = fullfiled_requests[fullfiled_requests['customer_id'].isin(fulfilled_bookings_count[fulfilled_bookings_count['count'] == 1]['customer_id'])].groupby('customer_id').max()
last_booking_date

,request_id,city,request_date,hours,price_eur,status,cleaner_id,rating,fulfilled,canc,month
customer_id,,,,,,,,,,,
C000007,R0000007,Paris,2025-01-21,3,81.0,fulfilled,P00020,2.0,True,False,2025-01-01
C000011,R0000011,Paris,2025-01-05,2,54.0,fulfilled,P00001,3.0,True,False,2025-01-01
C000015,R0000015,Paris,2025-01-03,2,54.0,fulfilled,P00003,5.0,True,False,2025-01-01
C000019,R0000019,Paris,2025-01-19,3,81.0,fulfilled,P00006,5.0,True,False,2025-01-01
C000033,R0000033,Paris,2025-01-13,2,54.0,fulfilled,P00007,2.0,True,False,2025-01-01
...,...,...,...,...,...,...,...,...,...,...,...
C024627,R0132300,Lille,2026-06-20,4,108.0,fulfilled,P01389,2.0,True,False,2026-06-01
C024628,R0132301,Lille,2026-06-15,3,81.0,fulfilled,P01393,5.0,True,False,2026-06-01
C024630,R0132303,Lille,2026-06-28,3,81.0,fulfilled,P01080,4.0,True,False,2026-06-01


In [17]:
#count the number of clients with only one fulfilled booking and a last booking date after 2026-05-01
last_booking_date[last_booking_date['month'] > '2026-05-01'].shape[0]


1619

In [18]:
#count the number of clients with only one fulfilled booking and a last booking date after 2026-06-01
last_booking_date[last_booking_date['month'] > '2026-06-01'].shape[0]

0

We can consider that since all may 2026 first time clients did not book anything in june, and that no one had its first booking in june, that those clients are lost.

In [23]:
#calculate proportion of retended customers
proportion_retained = fulfilled_bookings_distribution[fulfilled_bookings_distribution['count'] > 1]['number_of_customers'].sum() / fulfilled_bookings_distribution["number_of_customers"].sum() * 100
print(proportion_retained)

67.72142683814609


What to do next :
- referal : number of customers brought by the previous one 
- check if the compagny does reduction
- time between two commands
- is refferal efficient (does it help to retend new clients)